# Lesson 2: LCEL, Prompt Templates & Multi-Provider

## WHY
In Lesson 1 you built messages by hand. That works, but in production you'll have:  
- **Prompt templates** with placeholders (e.g. the user's message, server rules)  
- **Chains** that pipe a template → model → output parser in one line  
- **Multiple LLM providers** — maybe OpenAI for moderation and Anthropic for Q&A  

LangChain's **LCEL** (LangChain Expression Language) and `ChatPromptTemplate` solve all three.

**By the end of this notebook you will:**
1. Build chains using the LCEL `|` (pipe) operator
2. Create reusable prompt templates with `ChatPromptTemplate`
3. Add few-shot examples to guide the LLM's behavior
4. Swap from OpenAI to Anthropic without changing your chain logic
5. Package chain builders as importable functions

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print("OpenAI key loaded:", "OPENAI_API_KEY" in os.environ)
print("Anthropic key loaded:", "ANTHROPIC_API_KEY" in os.environ)

## WHAT — LCEL (the Pipe Operator)

LCEL lets you compose LangChain components with `|` — just like Unix pipes.  
Each component implements `.invoke()`, and the output of one flows into the next:

```
prompt | model | output_parser
```

This creates a **chain** — a single object you can `.invoke()` with your input variables.

📖 [LCEL docs](https://python.langchain.com/docs/concepts/lcel/)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Prompt template — {topic} is a placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human", "Explain {topic} in one sentence."),
])

# 2. Model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3. Output parser — extracts .content from AIMessage → plain string
parser = StrOutputParser()

# 4. Chain them with LCEL
chain = prompt | llm | parser

# 5. Invoke with the template variable
result = chain.invoke({"topic": "LangChain"})
print(result)

### What just happened?

1. `prompt.invoke({"topic": "LangChain"})` → fills the template → produces a `ChatPromptValue` (list of messages)  
2. `llm.invoke(messages)` → sends to OpenAI → returns an `AIMessage`  
3. `parser.invoke(ai_message)` → extracts `.content` → returns a plain `str`  

The `|` operator wires these steps together. You get `.invoke()`, `.stream()`, `.batch()`, and async variants for free.

## WHAT — `ChatPromptTemplate`

`ChatPromptTemplate` builds message lists from templates with `{placeholders}`.  
Two ways to create one:

| Method | When to use |
|--------|------------|
| `ChatPromptTemplate.from_messages([(role, template), ...])` | Multi-message (system + human) — **most common** |
| `ChatPromptTemplate.from_template("...")` | Single human message (quick prototyping) |

📖 [ChatPromptTemplate reference](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html)

In [ ]:
# Template with multiple variables
moderation_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Discord server moderator.\n"
     "Server rules:\n{rules}\n"
     "Respond with ALLOW or WARN and a brief reason."),
    ("human", "Check this message: {message}"),
])

# See what the template looks like when filled
filled = moderation_prompt.invoke({
    "rules": "1. Be respectful\n2. No spam\n3. No NSFW",
    "message": "Hey everyone! Check out my YouTube channel!!!",
})

# .to_messages() shows the actual message objects
for msg in filled.to_messages():
    print(f"[{msg.type}]\n{msg.content}\n")

In [ ]:
# Now run the full chain
moderation_chain = moderation_prompt | llm | StrOutputParser()

result = moderation_chain.invoke({
    "rules": "1. Be respectful\n2. No spam\n3. No NSFW",
    "message": "Hey everyone! Check out my YouTube channel!!!",
})

print(result)

## WHAT — Few-Shot Prompting

Few-shot prompting means giving the LLM **examples** of input → output before the real input.  
This is the single most effective way to control output format without fine-tuning.

For moderation, few-shot examples teach the LLM your server's specific standards — things a generic model might miss (e.g., "subtle self-promotion").

In the chat message format, examples are pairs of `("human", ...)` and `("ai", ...)` messages.

In [ ]:
# Few-shot examples as (human, ai) pairs in the prompt
fewshot_moderation_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Discord content moderator.\n"
     "Classify each message as ALLOW, WARN, or BAN.\n"
     "Respond with the label and a one-line reason."),

    # Example 1: friendly message → ALLOW
    ("human", "Check this message: 'Great job on the project!'"),
    ("ai", "ALLOW — Positive, supportive message."),

    # Example 2: borderline spam → WARN
    ("human", "Check this message: 'Follow my Instagram @cooluser for sick content'"),
    ("ai", "WARN — Self-promotion / unsolicited advertising."),

    # Example 3: clearly hostile → BAN
    ("human", "Check this message: 'You're all stupid and I hope this server dies'"),
    ("ai", "BAN — Targeted hostility and server disruption."),

    # The actual input
    ("human", "Check this message: '{message}'"),
])

fewshot_chain = fewshot_moderation_prompt | llm | StrOutputParser()

# Test with a new message the model hasn't seen
result = fewshot_chain.invoke({"message": "Anyone want to join my free Discord Nitro giveaway?"})
print(result)

In [ ]:
# Test a few more messages to see how consistent the few-shot examples make it
test_messages = [
    "Can someone help me with Python async/await?",
    "This server is trash, the mods don't do anything",
    "Hey check out my SoundCloud, link in bio!",
    "Thanks for the help everyone, really appreciate it!",
]

for msg in test_messages:
    result = fewshot_chain.invoke({"message": msg})
    print(f"Message: {msg}")
    print(f"Verdict: {result}\n")

## WHAT — Multi-Provider (Swapping Models)

One of LangChain's biggest wins: **chains are model-agnostic**.  
Because `ChatOpenAI` and `ChatAnthropic` both implement the same `BaseChatModel` interface, you can swap them without changing your prompt or parser.

Why this matters in production:
- **Cost**: Route cheap tasks to `gpt-4o-mini`, complex ones to Claude Sonnet  
- **Redundancy**: If OpenAI is down, fall back to Anthropic  
- **Evaluation**: Compare model performance on the same prompts

> **Note:** You need an `ANTHROPIC_API_KEY` in your `.env` for this section.  
> Get one at [console.anthropic.com](https://console.anthropic.com/). If you don't have one yet, read through the code — the pattern is the important part.

In [ ]:
from langchain_anthropic import ChatAnthropic

# Same interface as ChatOpenAI — model name and temperature
anthropic_llm = ChatAnthropic(
    model="claude-sonnet-4-20250514",
    temperature=0,
)

print(type(anthropic_llm))

In [ ]:
# Reuse the EXACT same prompt and parser — only swap the model
anthropic_chain = fewshot_moderation_prompt | anthropic_llm | StrOutputParser()

# Same test message, different model
result_openai = fewshot_chain.invoke({"message": "Free Nitro giveaway in DMs!"})
result_anthropic = anthropic_chain.invoke({"message": "Free Nitro giveaway in DMs!"})

print(f"OpenAI:    {result_openai}")
print(f"Anthropic: {result_anthropic}")

In [ ]:
# Side-by-side comparison on all test messages
print(f"{'Message':<55} {'OpenAI':<40} {'Anthropic'}")
print("=" * 140)

for msg in test_messages:
    r_openai = fewshot_chain.invoke({"message": msg})
    r_anthropic = anthropic_chain.invoke({"message": msg})
    # Truncate for display
    print(f"{msg:<55} {r_openai:<40} {r_anthropic}")

## Importable Chain Builders

These functions are ready to copy into the bot codebase.  
Notice: the chain-building functions don't know about Discord at all — that's intentional. The bot code will call these and wire the results into Discord responses.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import Runnable


def build_moderation_chain(
    llm: ChatOpenAI | None = None,
    rules: str = "Be respectful. No spam. No NSFW.",
) -> Runnable:
    """Build a moderation chain with few-shot examples.

    Parameters
    ----------
    llm : ChatOpenAI | None
        Chat model to use. Defaults to gpt-4o-mini with temperature=0.
    rules : str
        Server rules to include in the system prompt.

    Returns
    -------
    Runnable
        A chain that accepts {"message": str} and returns a str verdict.
    """
    if llm is None:
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         f"You are a Discord content moderator.\n"
         f"Server rules: {rules}\n"
         f"Classify each message as ALLOW, WARN, or BAN.\n"
         f"Respond with the label and a one-line reason."),
        ("human", "Check this message: 'Great job on the project!'"),
        ("ai", "ALLOW — Positive, supportive message."),
        ("human", "Check this message: 'Follow my Insta @cooluser'"),
        ("ai", "WARN — Self-promotion / unsolicited advertising."),
        ("human", "Check this message: '{message}'"),
    ])

    return prompt | llm | StrOutputParser()


def build_qa_chain(
    llm: ChatOpenAI | None = None,
    system_prompt: str = "You are a helpful Discord bot. Be concise.",
) -> Runnable:
    """Build a simple Q&A chain.

    Parameters
    ----------
    llm : ChatOpenAI | None
        Chat model to use. Defaults to gpt-4o-mini with temperature=0.7.
    system_prompt : str
        System instructions for the assistant.

    Returns
    -------
    Runnable
        A chain that accepts {"question": str} and returns a str answer.
    """
    if llm is None:
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{question}"),
    ])

    return prompt | llm | StrOutputParser()

In [ ]:
# Test the importable functions
mod_chain = build_moderation_chain()
qa_chain = build_qa_chain()

print("Moderation:")
print(mod_chain.invoke({"message": "Buy my NFTs at cheap prices!!!"}))

print("\nQ&A:")
print(qa_chain.invoke({"question": "What is a cog in py-cord?"}))

## Summary

| Concept | Key takeaway |
|---------|-------------|
| LCEL `\|` | Pipe components: `prompt \| model \| parser` → single `.invoke()` call |
| `ChatPromptTemplate` | Reusable message templates with `{placeholders}` |
| Few-shot | Add `(human, ai)` example pairs before the real input to control output format |
| Multi-provider | Swap `ChatOpenAI` ↔ `ChatAnthropic` — same chain, different model |
| `StrOutputParser` | Extracts `.content` from `AIMessage` → plain string |

**Next up → Lesson 3:** Custom tools with `@tool` and Pydantic schemas — teaching the LLM to *do things* (like mute a user or check server rules).